In [39]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
print(f"Device: {device}")

BASE_DIR = "./simpsons_dataset"

Device: mps


In [40]:
good_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

bad_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.05, 0.15)),
    transforms.RandomRotation(60),
    transforms.Grayscale(num_output_channels=3),
    transforms.GaussianBlur(kernel_size=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [41]:
full_ds = ImageFolder(root=BASE_DIR, transform=good_tf)
num_classes = 5
print(f"Total classes: {len(full_ds.classes)}")
print(f"Using classes: {full_ds.classes[:num_classes]}")

valid_targets = set(range(num_classes))
filtered_indices = [i for i, target in enumerate(full_ds.targets) if target in valid_targets]
print(f"Selected {len(filtered_indices)} images.")

full_ds_filtered = Subset(full_ds, filtered_indices)
full_ds_bad = ImageFolder(root=BASE_DIR, transform=bad_tf)
full_ds_bad_filtered = Subset(full_ds_bad, filtered_indices)

indices = np.random.permutation(len(full_ds_filtered))
train_size = int(0.8 * len(full_ds_filtered))
train_idx, val_idx = indices[:train_size], indices[train_size:]

Total classes: 43
Using classes: ['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson']
Selected 3026 images.


In [42]:
train_set = Subset(full_ds_filtered, train_idx)
val_set = Subset(full_ds_filtered, val_idx)
train_set_bad = Subset(full_ds_bad_filtered, train_idx)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=32, num_workers=0)
train_loader_bad = DataLoader(train_set_bad, batch_size=32, shuffle=True, num_workers=0)

In [43]:
def create_model():
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    for param in model.features.parameters():
        param.requires_grad = False
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(device)

In [44]:
def train_one(model, loader, criterion, optimizer, scheduler=None, epochs=3, name=""):
    model.train()
    for epoch in range(epochs):
        correct, total = 0, 0
        running_loss = 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
            running_loss += loss.item()
        if scheduler:
            scheduler.step()
        acc = 100 * correct / total
        print(f"{name} Ep {epoch+1}: Loss={running_loss/len(loader):.4f}, Acc={acc:.2f}%")

def validate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

In [45]:
criterion = nn.CrossEntropyLoss()

print("\nStepLR (Good augs)")
m1 = create_model()
opt1 = optim.Adam(filter(lambda p: p.requires_grad, m1.parameters()), lr=5e-4)
sch1 = optim.lr_scheduler.StepLR(opt1, step_size=2, gamma=0.1)
train_one(m1, train_loader, criterion, opt1, sch1, epochs=3, name="StepLR")
acc1 = validate(m1, val_loader)

print("\nCosineAnnealing (Good augs)")
m2 = create_model()
opt2 = optim.Adam(filter(lambda p: p.requires_grad, m2.parameters()), lr=5e-4)
sch2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=3)
train_one(m2, train_loader, criterion, opt2, sch2, epochs=3, name="Cosine")
acc2 = validate(m2, val_loader)

print("\nBad augs")
m3 = create_model()
opt3 = optim.Adam(filter(lambda p: p.requires_grad, m3.parameters()), lr=5e-4)
train_one(m3, train_loader_bad, criterion, opt3, epochs=3, name="BadAugs")
acc3 = validate(m3, val_loader)

print(f"StepLR Accuracy: {acc1}%")
print(f"Cosine Accuracy: {acc2}%")
print(f"Bad Augs Accuracy: {acc3}%")
print(f"Drop: {acc1 - acc3}%")


StepLR (Good augs)
StepLR Ep 1: Loss=1.1247, Acc=54.34%
StepLR Ep 2: Loss=0.8578, Acc=73.88%
StepLR Ep 3: Loss=0.7751, Acc=78.06%

CosineAnnealing (Good augs)
Cosine Ep 1: Loss=1.1146, Acc=56.28%
Cosine Ep 2: Loss=0.8722, Acc=73.60%
Cosine Ep 3: Loss=0.8039, Acc=75.17%

Bad augs
BadAugs Ep 1: Loss=1.2297, Acc=44.71%
BadAugs Ep 2: Loss=1.1676, Acc=48.64%
BadAugs Ep 3: Loss=1.1386, Acc=50.17%
StepLR Accuracy: 77.88778877887789%
Cosine Accuracy: 77.06270627062706%
Bad Augs Accuracy: 45.21452145214521%
Drop: 32.67326732673268%
